# QICK / tProc Tutorial — From Zero to Custom Pulses & Readout

This notebook is a from-scratch introduction to **QICK** (Quantum Instrumentation
Control Kit), the RFSoC-based control system in `src/libraries/qick`, written for
anyone in the group who hasn't used it before. It covers:

1. What QICK/RFSoC actually is, physically (the hardware architecture)
2. What the **tProc** is and why it exists
3. The Python software stack (`QickConfig`, `AbsQickProgram`, `AcquireMixin`, `QickProgram`)
4. Connecting to a board (locally, or remotely over Ethernet via Pyro4)
5. Configuring a signal generator (DAC output) and writing pulses
6. Configuring a readout (ADC input) and acquiring data
7. A full worked experiment (frequency sweep)
8. Using a DAC channel to characterize the board's RF output power (dBm/V)
9. Glossary + cheat sheet for quick reference

**Convention used below:** code cells marked **⚠️ requires hardware** need a real,
connected QICK board (or a Pyro4 proxy to one) to execute — they are written out
fully and correctly, but are *not* run in this notebook. Code cells marked
**✅ runs anywhere** are plain Python and can be executed right now, with no board,
to build intuition.


## 1. What QICK actually is

QICK runs on a **Xilinx/AMD RFSoC** (RF System-on-Chip): a single chip that combines
three very different kinds of hardware:

```
RFSoC die
│
├── PS (Processing System)  — hardened ARM Cortex-A53, runs Linux (PYNQ) + your Python
│     ▲
│     │ Ethernet (your PC / lab network reaches the board here)
│
├── PL (Programmable Logic) — FPGA fabric, reconfigurable via a bitstream (.bit file)
│     ├── tProc                  (soft CPU — sequences everything below, in real time)
│     ├── signal generators       (DDS + envelope playback)
│     └── readout / decimation    (downconversion, filtering, accumulation)
│
└── RF-DC tiles — hardened, FIXED silicon. NOT part of the FPGA fabric / PL.
      ├── DAC tiles  ← driven by the PL's signal generators, over AXI-Stream
      └── ADC tiles  → feed the PL's readout/decimation chains, over AXI-Stream
            (both tiles connect out to the board's physical SMA connectors)
```

Note the indentation: PS, PL, and the RF-DC tiles are three **siblings** — the RF-DC
tiles sit next to the FPGA fabric on the die, wired to it over an internal streaming bus,
but they are not themselves FPGA logic and are not reconfigured when you load a bitstream
(only PS-adjacent PL circuitry is).

Three regions, three different "programmability" levels:

- **RF-DC tiles** — hardened, fixed silicon (like a dedicated ADC/DAC chip baked into
  the die). Not reconfigurable at all from software; this is where the physical DAC/ADC
  SMA connectors ultimately are.
- **PL (FPGA fabric)** — reconfigurable *hardware*: a bitstream (`.bit` file) describes
  circuits — the tProc, the signal generators, the readout/decimation chains — that get
  synthesized into the fabric. Changing this requires HDL + Vivado, not Python. Your repo
  actually ships several pre-built bitstreams for different boards, e.g.:
  `src/libraries/qick/qick_lib/qick/qick_216.bit`, `qick_111.bit`, `qick_4x2.bit`
  (and the HDL source projects, if you ever need to modify the firmware itself, live
  under `src/libraries/qick/firmware/projects/`).
- **PS (ARM processor)** — a conventional, hardened CPU running embedded Linux (PYNQ)
  and your Python. This is what you actually talk to over Ethernet.

**Key takeaway:** from the QICK Python library alone (everything we do in this notebook),
you are only ever programming the tProc (layer inside the PL) — you cannot add new DAC
channels or change buffer sizes; that requires a different bitstream, a hardware-design task.


## 2. What the tProc is, and why there are *two* processors

The **tProc** ("tile processor" / timing processor) is a small, custom CPU — described in
HDL and synthesized into the FPGA fabric, exactly like the ARM core is hardened silicon,
except the tProc is "soft" (built from reconfigurable logic instead of fixed transistors).
It has its own registers, program memory, and a tiny fixed instruction set (see
`QickProgram.instructions` in `asm_v1.py` for the full list: `regwi`, `mathi`, `loopnz`,
`memwi`, `synci`, `pulse`-related `set`/`seti`, etc.).

### Why not just do everything in Python on the ARM core?

Because Python running under embedded Linux has **unpredictable timing** — OS scheduling,
interrupts, garbage collection — completely unacceptable when you need to trigger a pulse
at an *exact* clock cycle relative to a previous readout (as in almost every qubit
experiment: Ramsey, echo, randomized benchmarking...).

So QICK splits the work into a **control plane / data plane** hierarchy:

| Layer | Runs on | Speed / determinism | Job |
|---|---|---|---|
| Your experiment script | ARM PS (Python, Linux) | slow, non-deterministic (ms-scale jitter OK) | compile a program once, load it, start it, stream results back |
| tProc | FPGA fabric (soft CPU) | deterministic, fixed cycles/instruction (~100s of MHz) | sequence *when* things happen: loop, wait, trigger a pulse/readout |
| DDS / envelope playback / decimation | FPGA fabric (dedicated pipelines) | full sample rate (GSa/s) | actually synthesize/digitize every RF sample |

The tProc never touches individual RF samples — it just tells the dedicated generator/readout
hardware "start now with these parameters," and that hardware streams at full line rate on
its own, autonomously, until told otherwise. This is why you don't lose FPGA speed: the fast
data path runs independently of both CPUs once triggered; the tProc only handles the
comparatively low-rate control/timing logic, and Python only handles the even-lower-rate
"compile and launch" step.


## 3. The Python software stack (where every method actually lives)

All of the classes below are in `src/libraries/qick/qick_lib/qick/`.

```
qick_asm.py                                  (tProc-version-agnostic)
├── QickConfig                                board metadata + unit conversions
│                                              (freq2reg, us2cycles, get_maxv, description()...)
├── AbsQickProgram(ABC)                       declarations only: declare_gen, declare_readout,
│                                              add_envelope/add_gauss/add_DRAG..., config_all(), run()
│                                              compile() is @abstractmethod (version-specific!)
└── AcquireMixin                              acquire(), acquire_decimated(), run_rounds(),
                                               the round-stepping state machine, data reshaping

asm_v1.py                                     (tProc v1 specific)
├── QickProgram(AbsQickProgram)                the `instructions` dict (real ASM opcodes),
│                                              regwi/mathi/loopnz/... (auto-generated, see §3.1),
│                                              set_pulse_registers(), pulse(), compile()
└── AcquireProgram(AcquireMixin, QickProgram)   the class you actually inherit from

averager_program.py                           (user-facing templates, all built on AcquireProgram)
├── AveragerProgram        — 1D: loop over `reps` in hardware
├── RAveragerProgram       — 2D: `expts` (real-time HW sweep) × `reps`
├── NDAveragerProgram      — N-D: arbitrary stack of hardware-swept registers (QickSweep)
```

**Two totally different things live inside `AbsQickProgram`/`QickProgram`, and it's worth
being explicit about the distinction:**

- **Declarations** (`declare_gen`, `declare_readout`, `add_envelope`, ...) are pure Python
  bookkeeping — they compute register values and stash them in dicts (`self.gen_chs`,
  `self.ro_chs`, `self.envelopes`). **No ASM is emitted.** This configures the DAC/ADC/mixer
  hardware directly (via `config_gens`/`config_readouts`/`load_envelopes`), completely
  separately from whatever program the tProc runs.
- **ASM instructions** (`regwi`, `mathi`, `loopnz`, `pulse()`, ...) become the actual binary
  program the tProc's CPU executes. These *do* touch `self.prog_list`, which `compile()`
  turns into `self.binprog` (a list of 64-bit machine words) that gets loaded into the
  tProc's program memory.


### 3.1 How `regwi(...)` can work without a `def regwi` anywhere

This is easy to miss and worth understanding once. `QickProgram.__getattr__`
(`asm_v1.py`, ~line 1304) does this:

```python
def __getattr__(self, a):
    if a in self.__class__.instructions:      # e.g. "regwi" is a key in the ISA dict
        return lambda *args: self.append_instruction(a, *args)
    elif a in self.__class__.soccfg_methods:   # e.g. "freq2reg" — forward to soccfg
        return getattr(self.soccfg, a)
    else:
        return object.__getattribute__(self, a)
```

`__getattr__` only fires when normal attribute lookup *fails* — i.e. there is no real
`regwi` method. When you call `self.regwi(0, rcount, 0)`, Python can't find `regwi`
normally, falls into `__getattr__`, sees `"regwi"` is a key in the `instructions` ISA
dictionary, and hands back a throwaway function that appends
`{'name': 'regwi', 'args': (0, rcount, 0)}` to `self.prog_list`. Every one of the ~20 ASM
mnemonics works this same way — one generic dispatch mechanism, not 20 hand-written methods.

The cell below is a **standalone, runnable** re-implementation of the same trick (no QICK,
no hardware needed) so you can convince yourself of the mechanism interactively.


In [ ]:
# ✅ runs anywhere — toy reimplementation of the QickProgram.__getattr__ trick

class ToyISA:
    # our tiny fake instruction set: name -> (nothing important here, just existence)
    instructions = {"add", "sub", "jump"}

    def __init__(self):
        self.prog_list = []

    def append_instruction(self, name, *args):
        self.prog_list.append({"name": name, "args": args})

    def __getattr__(self, name):
        if name in self.__class__.instructions:
            return lambda *args: self.append_instruction(name, *args)
        return object.__getattribute__(self, name)


prog = ToyISA()
prog.add(1, 2)      # "add" isn't a real method -> falls into __getattr__ -> dynamic lambda
prog.jump("LOOP")
print(prog.prog_list)


## 4. Connecting to the board

There are two ways to run a QICK program, and your repo has both set up
(`src/libraries/qick/pyro4/`):

**A. Local** — a Jupyter server running directly on the board's ARM PS. All Python
(including `compile()`) executes on-board; "remote" only describes how you view the
notebook in your browser.

```python
from qick import QickSoc
soc = QickSoc(bitfile="qick_4x2.bit")   # loads the FPGA bitstream (see src/libraries/qick/qick_lib/qick/*.bit)
soccfg = soc
```

**B. Remote (Pyro4)** — your script runs on your own PC; only remote-procedure calls
travel over Ethernet to a server process running on the board
(`src/libraries/qick/pyro4/pyro_service.py` starts this). Program *compilation* (all the
register math, `regwi`/`mathi`/`pulse()` calls, `compile()`) happens locally on your PC;
only `soc.load_bin_program(...)`, `soc.start_tproc()`, `soc.get_accumulated(...)`-style
calls actually go over the wire and execute against real hardware.

```python
from qick.pyro import make_proxy
soc, soccfg = make_proxy(ns_host="<board-or-nameserver-ip>", ns_port=8000, proxy_name="rfsoc")
```

Either way, once you have `soc` and `soccfg`, everything below is identical.


In [ ]:
# ⚠️ requires hardware — pick ONE of the two connection methods and run it.

# --- Option A: local (script/notebook running on the board itself) ---
# from qick import QickSoc
# soc = QickSoc(bitfile="qick_4x2.bit")   # match this to your actual board/firmware
# soccfg = soc

# --- Option B: remote over Ethernet via Pyro4 (script running on your PC) ---
# from qick.pyro import make_proxy
# soc, soccfg = make_proxy(ns_host="<ip-of-nameserver>", ns_port=8000, proxy_name="rfsoc")

# Once connected, this tells you exactly what hardware you have: how many DAC/ADC
# channels, their sample rates, and which physical SMA port each logical channel maps to.
print(soccfg)


**Reading `print(soccfg)`:** the output (built by `QickConfig.description()`) lists
every generator and readout channel with its sample rate, fabric clock, DDS bit width, and
envelope memory size, and — critically — a line like *"DAC tile 2, blk 0 is 0_230 on JHC3,
or QICK box DAC port 8"*. That mapping (`_describe_dac`/`_describe_adc` in `qick_asm.py`)
is how you find out which physical SMA connector corresponds to `gen_ch=0` in your code —
run this once for your board before doing any physical measurement (e.g. the power
characterization in §8).


## 5. The three program templates (`averager_program.py`)

Every QICK experiment is a subclass of one of these. You only ever write two methods:
`initialize()` (runs once) and `body()` (runs once per shot); the class wraps `body()` in
a hardware loop for you.

```
AveragerProgram            1D loop, hardware-repeats `reps` times
  LOOP_J (reps):
      body()

RAveragerProgram           2D loop: real-time hardware sweep of one variable + reps
  LOOP_I (expts):
      LOOP_J (reps):
          body()
      update()              <- e.g. increments a frequency register in hardware

NDAveragerProgram          N-D: reps (outer) + an arbitrary stack of QickSweep-swept registers
  LOOP_rep (reps):
      LOOP_sweep_N ... LOOP_sweep_0:
          body()
      <update each sweep>
```

Use `AveragerProgram` for a single point (you sweep from Python), `RAveragerProgram` when
you want one parameter swept *in hardware* (much faster — no per-point recompile/reload),
and `NDAveragerProgram` when you need more than one hardware-swept axis at once
(`add_sweep()` + `QickSweep`/`merge_sweeps`).


## 6. Configuring a signal generator (DAC output)

Two declarations control a DAC channel:

### `declare_gen(ch, nqz=1, mixer_freq=None, mux_freqs=None, mux_gains=None, mux_phases=None, ro_ch=None)`
Pure bookkeeping (no ASM). Key parameters:

| param | meaning |
|---|---|
| `ch` | generator channel index (see `soccfg['gens']`, and `print(soccfg)` for the physical port) |
| `nqz` | **Nyquist zone**: 1 for frequencies below fs/2, 2 for the next band up (fs/2 to fs). Increases output power in the 2nd/3rd Nyquist zone — pick based on where your target frequency actually falls. |
| `mixer_freq` | for channels with a digital mixer (interpolated/muxed generators): the NCO frequency mixed with the DDS output, letting you reach higher absolute frequencies than the DDS range alone |
| `mux_freqs/gains/phases` | only for multiplexed generators — multiple simultaneous CW tones out of one DAC channel |

### `set_pulse_registers(ch, style=..., freq=..., phase=..., gain=..., length=..., mode=..., outsel=..., waveform=...)`
This is tProc-v1-specific (writes actual ASM registers). Key parameters:

| param | meaning |
|---|---|
| `style` | `"const"` (flat pulse), `"arb"` (arbitrary envelope, needs `waveform=`), `"flat_top"` (envelope ramp + flat middle + ramp down) |
| `freq` | **register value**, not MHz — always compute with `self.freq2reg(f_MHz, gen_ch=ch)` |
| `phase` | register value — compute with `self.deg2reg(deg, gen_ch=ch)` |
| `gain` | signed integer DAC code; `self.get_maxv(ch)` is full scale (≈ ±32000, board-dependent) |
| `mode` | `"oneshot"` (default, plays once) or `"periodic"` (repeats continuously after `pulse()` — this is how you get a true CW tone, e.g. for RF power characterization, §8) |
| `outsel` | `"product"` (envelope × DDS, normal case), `"dds"` (DDS only, no envelope), `"input"`, `"zero"` |
| `waveform` | name of an envelope previously registered with `add_envelope`/`add_gauss`/`add_DRAG`/... |

### Envelopes (for `style="arb"`/`"flat_top"`)
`add_gauss(ch, name, sigma, length, ...)`, `add_DRAG(ch, name, sigma, length, delta, alpha, ...)`
(DRAG suppresses leakage to a qubit's second excited state), `add_cosine(...)`,
`add_triangle(...)`, or raw `add_envelope(ch, name, idata=..., qdata=...)`. All lengths are
in fabric clocks (tProc v1) — use `self.us2cycles(gen_ch=ch, us=...)` to convert from time.


In [ ]:
# ⚠️ requires hardware — a fully-commented, minimal generator-only program.
# This just plays one constant tone. Compare against §8 for a CW (periodic) version
# suitable for RF power measurement.

from qick.averager_program import AveragerProgram

class ToneProgram(AveragerProgram):
    def initialize(self):
        cfg = self.cfg

        # --- declare the DAC channel: bookkeeping only, no ASM emitted yet ---
        self.declare_gen(ch=cfg["gen_ch"], nqz=cfg.get("nqz", 1))

        # --- convert physical units (MHz) into register values ---
        freq_reg = self.freq2reg(cfg["freq_MHz"], gen_ch=cfg["gen_ch"])
        length_cycles = self.us2cycles(cfg["length_us"], gen_ch=cfg["gen_ch"])

        # --- write the pulse parameters into registers (this DOES touch prog_list) ---
        self.set_pulse_registers(
            ch=cfg["gen_ch"],
            style="const",
            freq=freq_reg,
            phase=0,
            gain=cfg["gain"],          # integer DAC code, up to self.get_maxv(cfg["gen_ch"])
            length=length_cycles,
        )

        self.synci(200)  # let the DAC/mixer settings settle before the first pulse

    def body(self):
        self.pulse(ch=self.cfg["gen_ch"])   # actually emit the pulse (schedules it)
        self.wait_all()                     # wait for all pending pulses/readouts to finish
        self.sync_all(self.us2cycles(1))    # pad 1 us before the next rep


# config = {
#     "gen_ch": 0,
#     "nqz": 1,
#     "freq_MHz": 1000,
#     "gain": 20000,          # DAC code; self.get_maxv(0) tells you the max for this channel
#     "length_us": 5,
#     "reps": 1000,
#     "style": "const",  # "const" or "periodic" (CW) are the only two styles supported by this program

# }
# prog = ToneProgram(soccfg, config)
# prog.run_rounds(soc)   # just play it, no readout/data collection


### 6.1 Envelopes: the 4 pre-defined shapes, and a fully custom one

`ToneProgram` above used `style="const"` — a flat-topped pulse, no envelope needed. For
anything shaped (Gaussian, DRAG, or literally anything you can write as a numpy array),
you need `style="arb"` (or `"flat_top"`) plus a named envelope registered beforehand.

Every one of `add_gauss`/`add_DRAG`/`add_cosine`/`add_triangle` is just a convenience
wrapper: it builds `idata`/`qdata` numpy arrays (via the plain-numpy functions in
`qick.helpers`) and hands them to `add_envelope(ch, name, idata=..., qdata=...)`, which
does the real work of registering the waveform (with a hard range check against
`get_maxv(ch)`, and a requirement that `len(idata)` be a multiple of the channel's
`samps_per_clk`). **Because `add_envelope` takes raw numpy arrays, you can build any shape
yourself, by hand, and register it the exact same way** — that's what "arbitrary" means
in `style="arb"`.

The cell below plots all four pre-defined shapes plus one made-up-by-hand shape, using the
same numpy functions QICK uses internally — **no hardware needed**, just to build
intuition for what each one looks like before you use it in a real pulse.


In [ ]:
# ✅ runs anywhere — visualize the pre-defined envelope shapes + a custom one.
# The predefined ones (cosine/gauss/triang/DRAG) are the exact same numpy functions
# add_gauss/add_DRAG/add_cosine/add_triangle call internally (qick.helpers).
#
# custom_shape() below is defined once, here, and reused as-is (not re-derived) by the
# hardware cell further down — same idea as the built-in helpers: a plain function of
# (length, maxv) that returns a numpy array, nothing hardware-specific about it.
import numpy as np
import matplotlib.pyplot as plt
from qick.helpers import cosine, gauss, triang, DRAG

def custom_shape(length, maxv):
    """Hand-built envelope: fast cosine rise, flat top, slow exponential decay.
    Not one of QICK's predefined shapes -- just plain numpy, written the same way
    you'd write any arbitrary shape of your own."""
    t = np.arange(length)
    rise, decay_start = length // 6, length // 2
    y = np.zeros(length)
    y[:rise] = maxv * (1 - np.cos(np.pi * t[:rise] / rise)) / 2   # cosine ramp up
    y[rise:decay_start] = maxv                                     # flat top
    tail = t[decay_start:] - decay_start
    y[decay_start:] = maxv * np.exp(-tail / (length / 8))          # exponential decay
    return y

length = 100      # samples
maxv = 30000      # stand-in for get_maxv(ch) — on real hardware, use that instead

shapes = {
    "cosine":   cosine(length=length, maxv=maxv),
    "gauss":    gauss(mu=length/2, si=length/8, length=length, maxv=maxv),
    "triangle": triang(length=length, maxv=maxv),
    "custom (hand-built)": custom_shape(length, maxv),
}
drag_i, drag_q = DRAG(mu=length/2, si=length/8, length=length, maxv=maxv,
                       delta=0.05, alpha=0.5, det=0)
shapes["DRAG (I)"] = drag_i
shapes["DRAG (Q)"] = drag_q

t = np.arange(length)
fig, ax = plt.subplots(figsize=(8, 4))
for name, y in shapes.items():
    ax.plot(t, y, label=name)
ax.set_xlabel("sample")
ax.set_ylabel("DAC code")
ax.legend()
ax.set_title("Envelope shapes: predefined vs. hand-built")


In [ ]:
# ⚠️ requires hardware — registering + using each envelope on a real generator channel.
# Extends ToneProgram (§6) to also load and play one custom + all four predefined shapes.
# Reuses custom_shape() from the cell above -- run that cell first (or paste the
# function in) so it's defined in this kernel session.

from qick.averager_program import AveragerProgram
import numpy as np

class EnvelopeProgram(AveragerProgram):
    def initialize(self):
        cfg = self.cfg
        ch = cfg["gen_ch"]
        self.declare_gen(ch=ch, nqz=cfg.get("nqz", 1))

        length = self.us2cycles(cfg["length_us"], gen_ch=ch)  # fabric-clock cycles

        # --- the 4 predefined shapes: each just wraps add_envelope() for you ---
        self.add_gauss(ch, "gauss_pulse", sigma=length/6, length=length)
        self.add_DRAG(ch, "drag_pulse", sigma=length/6, length=length,
                       delta=cfg["anharmonicity_MHz"], alpha=0.5)
        self.add_cosine(ch, "cos_pulse", length=length)
        self.add_triangle(ch, "tri_pulse", length=length)

        # --- a fully arbitrary envelope, using the SAME custom_shape() function from
        # the plotting cell above -- reused, not rewritten. add_envelope() only requires:
        # idata/qdata same length, len % samps_per_clk == 0, max(abs(...)) <= get_maxv(ch).
        samps_per_clk = self.soccfg['gens'][ch]['samps_per_clk']
        n = length * samps_per_clk
        idata = custom_shape(n, self.get_maxv(ch))
        qdata = np.zeros(n)   # real-valued envelope; nonzero qdata needs complex_env support

        self.add_envelope(ch, "custom_pulse", idata=idata, qdata=qdata)

        # --- pick ONE of the five to actually play (style="arb" needs waveform=name) ---
        freq_reg = self.freq2reg(cfg["freq_MHz"], gen_ch=ch)
        self.set_pulse_registers(
            ch=ch, style="arb",
            freq=freq_reg, phase=0, gain=cfg["gain"],
            waveform=cfg["waveform_name"],   # e.g. "gauss_pulse", "drag_pulse", "custom_pulse", ...
        )
        self.synci(200)

    def body(self):
        self.pulse(ch=self.cfg["gen_ch"])
        self.wait_all()
        self.sync_all(self.us2cycles(1))


# config = {
#     "gen_ch": 0, "nqz": 1, "freq_MHz": 1000, "gain": 20000,
#     "length_us": 0.1, "anharmonicity_MHz": 200,
#     "waveform_name": "custom_pulse",   # swap to try each shape
#     "reps": 1000,
# }
# prog = EnvelopeProgram(soccfg, config)
# prog.run_rounds(soc)


### 6.2 How an envelope actually becomes an analog signal -- and where it lives

Worth tracing the full pipeline once, precisely, because it resolves a natural but wrong
assumption: **the envelope shape is not stored in the tProc.**

#### The pipeline

1. **Python, once, before the tProc starts:** `add_envelope`/`add_gauss`/etc. build
   `idata`/`qdata` int16 arrays. `load_envelopes()` (part of `config_all()`) pushes these
   into a dedicated on-chip SRAM block that is part of **each generator's own IP core** --
   not the tProc, and not shared across channels. `print(soccfg)` shows each generator's
   envelope memory size directly (`"envelope memory: %d %s samples"`, from `gen['maxlen']`).
   This is a genuine, one-time ARM-to-FPGA memory write, done via `soc.load_envelope(ch,
   data, addr)` -- a completely different method from `soc.load_bin_program(...)`, which is
   what loads the tProc's own program memory. Two different memories, two different methods.
2. **`set_pulse_registers`/`pulse()` (tProc, per shot):** sends no waveform data at all --
   only a handful of small register values: `freq`, `phase`, `gain`, `addr` (the envelope's
   *start address*, a pointer), and `mode` (packs the length plus output-select flags).
   Checking `FullSpeedGenManager.write_regs` (`asm_v1.py`) shows exactly how each pulse
   style uses this:
   - **`style="const"`** -- `outsel` is hard-set to `"dds"`. It **never reads envelope
     memory at all** -- a pure DDS tone gated on for `length` cycles at `gain` amplitude.
   - **`style="arb"`** -- `addr` = the named waveform's stored address; the generator reads
     `wfm_length` samples starting there and multiplies each by the DDS output
     (`outsel="product"`).
   - **`style="flat_top"`** -- three segments from the *same* pre-loaded envelope: ramp-up
     (from `addr`), flat middle (pure DDS, half gain via a second `gain2` register), ramp-down
     (from `addr2 = addr + half-length`).
3. **The generator hardware, autonomously, at full sample rate:** once triggered, it reads
   envelope samples and runs its own DDS completely independently of the tProc, streaming
   the result to the RF-DC DAC tile, which produces the actual analog waveform at the SMA
   port. The tProc is not involved again until the next `pulse()` call.

#### Consequence for sweeps: register-only vs. shape changes

Because the envelope memory write happens exactly once, before the run starts, **sweeping
`freq`/`gain`/pulse `length` never touches it again** -- those are pure register operations
inside the tProc (§8/§8.1's `QickSweep`/`mathi`), fully hardware-sequenced, no ARM
involvement mid-run. But sweeping the *shape itself* (e.g. a Gaussian's sigma, or switching
pulse families) is not representable as a register value -- the array content genuinely
changes, so it requires a fresh `load_envelopes()` call from Python for every point:

```python
# Register-only sweep (fast, entirely in hardware) -- see §8/§8.1
freq_reg = self.get_gen_reg(ch, "freq")
self.add_sweep(QickSweep(self, freq_reg, start=90, stop=130, expts=41))

# Shape sweep (envelope content changes -> needs a Python-level loop, one reload per point)
sigmas_us = np.linspace(0.01, 0.1, 10)
results = []
for sigma in sigmas_us:
    class GaussPoint(AveragerProgram):
        def initialize(self):
            cfg = self.cfg
            self.declare_gen(ch=cfg["gen_ch"], nqz=1)
            self.add_gauss(cfg["gen_ch"], "g",
                            sigma=self.us2cycles(sigma, gen_ch=cfg["gen_ch"]),
                            length=cfg["length"])              # <- new idata every iteration
            self.set_pulse_registers(ch=cfg["gen_ch"], style="arb",
                                      freq=self.freq2reg(cfg["freq_MHz"], gen_ch=cfg["gen_ch"]),
                                      phase=0, gain=cfg["gain"], waveform="g")
        def body(self):
            self.pulse(ch=self.cfg["gen_ch"])
            self.wait_all(); self.sync_all(self.us2cycles(1))

    prog = GaussPoint(soccfg, {"gen_ch": 0, "freq_MHz": 1000, "gain": 20000,
                                "length": 100, "reps": 1000})
    results.append(prog.acquire(soc))   # each point: fresh compile + fresh load_envelopes()
```

**Decision rule:** does the swept quantity plug into `freq`/`phase`/`gain`/`length` (or which
pre-loaded shape to select via `addr`)? Use a hardware sweep -- nanoseconds between points.
Does it change the envelope *array itself*? Use a Python loop -- milliseconds between points
(full `config_all()` reload each time), fine for characterization, too slow for real-time
feedback.

#### Two different "nanosecond" claims -- don't conflate them

- **Triggering is fixed and fast:** the tProc executing `pulse()`'s `set` instruction is one
  tProc clock cycle -- check `print(soccfg)`'s core clock line for your board's actual
  number, but it's on the order of a couple of nanoseconds. This doesn't depend on your pulse.
- **Playback duration is *not* fixed -- it's whatever `length` you configured.** Once
  triggered, the generator plays out the envelope over your chosen duration (tens of ns to
  several us, your choice), with zero further tProc involvement during that window.
  Individual samples inside that pulse are emitted at the DAC's own sample rate (also
  sub-nanosecond, but a different, separate number from the tProc's clock).

### 6.3 Why `gain` alone doesn't tell you dBm

`gain` is a dimensionless DAC code (a fraction of full scale, via `get_maxv`). The actual
output voltage/power at the SMA connector depends on the DAC's analog full-scale current,
the output balun/network, board attenuation, and which Nyquist zone you're in — none of
which is fully captured in the firmware config. **You cannot compute dBm purely from
`gain` and a datasheet number reliably; it has to be measured.** See §9 for the actual
characterization workflow.


## 7. Configuring a readout (ADC input)

### `declare_readout(ch, length, freq=None, phase=0, sel='product', gen_ch=None, weights=None, edge_counting=False, ...)`
Also pure bookkeeping. Key parameters:

| param | meaning |
|---|---|
| `ch` | readout channel index (index into `soccfg['readouts']`) |
| `length` | readout window length, in decimated samples (tProc v1) |
| `freq` | downconversion frequency (MHz) — usually matches the drive/readout pulse frequency |
| `gen_ch` | pass the paired generator channel so the readout's downconversion frequency gets **exactly** frequency-matched (bit-for-bit) to it — critical for stable, driftless demodulated I/Q |
| `sel` | `"product"` (downconverted product, normal), `"dds"`, `"input"` (raw ADC, no downconversion) |
| `weights` | optional per-sample IQ array — a matched filter multiplied against the readout window before summing (used for optimal qubit-state discrimination) |
| `edge_counting` | single-photon-style discrete pulse counting mode, if the hardware supports it |

In `body()`, you call `self.trigger(ros=[ro_ch], ...)` (or, in the averager templates, this
is often wired up for you) to arm the ADC capture window in time with your pulses.

### Getting data back
- **`acquire(soc, ...)`** — hardware-accumulated (summed) I/Q, averaged over `reps`/`rounds`.
  Optionally applies a real-time hardware `threshold`/`angle` for single-shot qubit-state
  discrimination.
- **`acquire_decimated(soc, ...)`** — raw (not summed) downconverted ADC samples: your
  "scope trace." Slower, but essential for lining up pulses or doing loopback
  characterization (§8).


In [ ]:
# ⚠️ requires hardware — generator + readout loopback: play a tone, digitize it back.

from qick.averager_program import AveragerProgram

class LoopbackProgram(AveragerProgram):
    def initialize(self):
        cfg = self.cfg

        self.declare_gen(ch=cfg["gen_ch"], nqz=1)
        self.declare_readout(
            ch=cfg["ro_ch"],
            length=cfg["readout_length"],
            freq=cfg["freq_MHz"],
            gen_ch=cfg["gen_ch"],   # frequency-match RO to the generator
        )

        freq_reg = self.freq2reg(cfg["freq_MHz"], gen_ch=cfg["gen_ch"], ro_ch=cfg["ro_ch"])
        self.set_pulse_registers(
            ch=cfg["gen_ch"], style="const",
            freq=freq_reg, phase=0, gain=cfg["gain"],
            length=self.us2cycles(cfg["pulse_length_us"], gen_ch=cfg["gen_ch"]),
        )
        self.synci(200)

    def body(self):
        # arm the ADC capture window, then play the pulse
        self.trigger(ros=[self.cfg["ro_ch"]], pins=[], adc_trig_offset=self.cfg["adc_trig_offset"])
        self.pulse(ch=self.cfg["gen_ch"])
        self.wait_all()
        self.sync_all(self.us2cycles(1))


# config = {
#     "gen_ch": 0, "ro_ch": 0,
#     "freq_MHz": 100, "gain": 20000,
#     "pulse_length_us": 2, "readout_length": 400,
#     "adc_trig_offset": 100,   # cycles between trigger and ADC start, to compensate cable/latency delay
#     "reps": 100, "soft_avgs": 1,
# }
# prog = LoopbackProgram(soccfg, config)
# iq_trace = prog.acquire_decimated(soc)     # raw scope-like trace, one array per RO channel
#
# import matplotlib.pyplot as plt
# t = prog.get_time_axis(ro_index=0)         # time axis in us
# plt.plot(t, iq_trace[0][:, 0], label="I")
# plt.plot(t, iq_trace[0][:, 1], label="Q")
# plt.xlabel("time (us)"); plt.ylabel("ADC code"); plt.legend()


## 7.1 Where does the data actually go, and when do you get it back?

Worth being precise about this, since "the tProc runs the sweep" glosses over several
different storage locations that are easy to conflate.

### The tProc has three separate storage areas — none of them "RAM" in the general sense

- **Register file** — small, fast, per-page scratch slots wired directly into the tProc's
  math unit. `regwi`/`mathi` operate here — loop counters, and (for `RAveragerProgram`) the
  swept frequency register itself, all live here.
- **Data memory (dmem)** — a *separate* memory block, accessed via different instructions
  (`memwi`/`memri`). You've actually already seen this: `AveragerProgram.make_program()`
  has `p.memwi(0, rcount, self.COUNTER_ADDR)` — writing the shot counter out to a **data
  memory** address specifically so Python can poll it from outside while the tProc keeps
  running. `print(soccfg)` shows this block's size ("data memory %d words").
- **Program memory (pmem)** — holds the compiled ASM binary itself. Loaded once before
  `start_tproc()`; not touched as data during a run.

### Measurement results don't go into any of those — they go into a separate, small,
### *circular* hardware buffer

Each readout channel has its own **accumulated buffer**, part of the readout/averaging IP
block (not the tProc at all), sized `avg_maxlen` samples — also visible in `print(soccfg)`.
For anything but a short run, this buffer is *smaller* than your total shot count, and it's
circular: `qick_lib/qick/streamer.py` computes the write address as
`addr = last_shots * reads_per_count[iCh] % avg_maxlen` — it wraps around. So it physically
cannot hold an entire long sweep; something has to continuously drain it while the tProc
is still running, or new shots overwrite unread old ones.

### That "something" is a background thread, running on the ARM PS, concurrently with the tProc

When you call `acquire()`, a background Python thread (`streamer.py`, `_run_readout()`)
does this in a loop, **while the tProc is still executing your sweep**:

1. Poll `soc.get_tproc_counter(addr=counter_addr)` — reads the tProc's data-memory
   progress counter (the same one `memwi` writes every shot).
2. Once a "stride" worth of new shots has completed (~10% of the buffer size — deliberately
   conservative, to stay well ahead of the buffer wrapping around), read that chunk directly
   out of the hardware buffer with `soc.get_accumulated(ch=ch, address=addr, length=newpoints)`
   and push it onto a queue.
3. Your foreground code (`poll_data()`, called from `AcquireMixin.finish_round()`) pulls
   chunks off that queue as they arrive.

**Important subtlety: `get_accumulated()` is a plain read — it does not clear or reset the
buffer.** The hardware buffer's write side is completely independent: the readout hardware
keeps writing each new shot's result into the next slot, wrapping around
(`address = shot_index % avg_maxlen`), *whether or not* anyone has read the old contents of
that slot yet. So "the background thread drains the buffer" doesn't mean the buffer becomes
empty — it means the thread is copying each slot's contents into Python *before* the
hardware's write pointer wraps back around and overwrites that same slot with a later shot.
Nothing is ever "emptied"; a slot's old data is only lost once new hardware writes land on
top of it. This is exactly what the overflow check further down is guarding against — "you
were about to lap the buffer before finishing that read."

So results move from FPGA hardware into Python **incrementally, throughout the sweep** —
not "wait for the whole thing to finish, then transfer." It's a producer/consumer pipeline:
tProc (writing new shots into the circular buffer) → background thread (racing to copy each
chunk out before it gets overwritten, into a software queue) → your acquisition code
(consuming the queue) — all running concurrently, entirely on the ARM PS. The only network
hop, if any, is the Pyro4 one from §4: the *final*, already-averaged numpy array gets shipped
to your PC over Ethernet, once, at the very end.

### What happens if draining falls behind

`streamer.py` guards this explicitly:

```python
if newpoints >= avg_maxlen:
    raise RuntimeError("Overflowed the averages buffer (%d unread samples >= buffer size %d)."
                        "\nYou need to slow down the tProc by increasing relax_delay.")
```

A real circular-buffer overflow: the tProc never waits for permission to keep writing, so if
the background thread can't keep up (progress bar overhead, a GC pause, whatever), new
accumulated shots overwrite unread ones. This is part of why some experiment configs add
deliberate padding (`relax_delay`-style) between shots — to give the streaming thread room
to keep up.

### Every storage location, in one place

Combining this with §6.2 (envelope memory), here is the complete picture -- five distinct
hardware blocks, each doing exactly one job. The tProc itself only ever holds small numbers;
anything wide (waveform samples going out, measurement samples coming back) lives in a
separate, dedicated memory block belonging to the generator or readout hardware, never the
tProc:

| Block | Belongs to | Holds | Written by |
|---|---|---|---|
| Register file | tProc | loop counters, freq/gain/addr/phase — small scratch values | ASM (`regwi`/`mathi`), every shot |
| Data memory (dmem) | tProc | the shot progress counter (`COUNTER_ADDR`) | ASM (`memwi`), every shot |
| Program memory (pmem) | tProc | the compiled ASM binary itself | `soc.load_bin_program()`, once |
| Envelope memory (§6.2) | each generator IP core | pulse shapes (`idata`/`qdata`) | `soc.load_envelope()`, once |
| Accumulated / decimated buffer | each readout IP core | measurement results | readout hardware, continuously; drained by the background thread, above |

### Putting it together: two diagrams of the same explanation

The first shows the **spatial layout** — which hardware/software component talks to which,
and over what path. The second shows the **concurrent timeline** — the tProc, the
background thread, and your foreground code are all running *at the same time*, not in a
simple request/response sequence.


```mermaid
flowchart LR
    subgraph ARM["ARM PS -- embedded Linux + your Python"]
        direction TB
        ACQ["your code:\nprog.acquire(soc)"]
        THREAD["background thread\nstreamer._run_readout()"]
        QUEUE[("data_queue")]
        ACQ -->|"start_readout()"| THREAD
        THREAD -->|"push data chunks"| QUEUE
        QUEUE -->|"poll_data()"| ACQ
    end

    subgraph PL["PL -- FPGA fabric"]
        direction TB
        TPROC["tProc\nregisters + data memory + program memory\nruns the compiled ASM"]
        GEN["signal generator\nDDS + envelope playback"]
        RO["readout / decimation"]
        BUF[("accumulated buffer\ncircular, size = avg_maxlen")]
        DMEM[("tProc data memory\nCOUNTER_ADDR")]
        TPROC -->|"pulse():\n'set' instr, freq/gain snapshot"| GEN
        TPROC -->|"trigger():\narm capture window"| RO
        RO -->|"writes each new shot,\nwraps around on its own\n(no clear on read)"| BUF
        TPROC -.->|"memwi:\nshot counter"| DMEM
    end

    subgraph RFDC["RF-DC tiles -- hardened silicon"]
        direction TB
        DAC["DAC tile"]
        ADC["ADC tile"]
    end

    GEN --> DAC --> SMA1["SMA out"]
    SMA2["SMA in"] --> ADC --> RO

    THREAD -->|"get_tproc_counter():\npoll progress"| DMEM
    THREAD -->|"get_accumulated():\nplain read, before hw\noverwrites this slot"| BUF

    ACQ -.->|"Pyro4 RPC over Ethernet\n(only if using the remote setup, section 4)"| PC["your PC\n(final averaged array)"]
```

```mermaid
sequenceDiagram
    participant Py as Your code (foreground)
    participant Thread as Background thread
    participant tProc as tProc (FPGA)
    participant Buf as Accumulated buffer

    Py->>tProc: config_all() -- load program, configure gens/readouts
    Py->>Thread: start_readout(total_shots)
    Thread->>tProc: start_tproc()

    loop until total_shots reached
        tProc->>tProc: run body(): pulse + trigger, one shot
        tProc->>Buf: write accumulated I/Q -- wraps around, overwrites oldest unread slot
        tProc->>tProc: memwi: increment shot counter (dmem)
        Thread->>tProc: get_tproc_counter() -- poll progress
        alt a stride worth of new shots is ready
            Thread->>Buf: get_accumulated() -- plain read, copies data out
            Note right of Buf: read does NOT clear the slot --<br/>it's only overwritten later,<br/>when the write pointer wraps back here
            Thread->>Thread: push chunk onto data_queue
        end
    end

    Py->>Thread: poll_data() -- pull chunks from queue
    Thread-->>Py: data chunks, as they arrive
    Py->>Py: _process_accumulated() / _average_buf() -- reshape, average

    Note over Py: if using Pyro4 (section 4), the final<br/>averaged array is shipped to your<br/>PC over Ethernet here, once
```

## 8. Full worked example: a real-time frequency sweep (`RAveragerProgram`)

This ties declarations, ASM, and acquisition together in one realistic experiment — e.g.
a resonator/qubit spectroscopy scan, swept **in hardware** (fast) rather than by
re-running Python per point (slow).


In [ ]:
# ⚠️ requires hardware — frequency sweep example.

from qick.averager_program import RAveragerProgram

class FreqSweepProgram(RAveragerProgram):
    def initialize(self):
        cfg = self.cfg
        self.declare_gen(ch=cfg["gen_ch"], nqz=1)
        self.declare_readout(ch=cfg["ro_ch"], length=cfg["readout_length"],
                              freq=cfg["start"], gen_ch=cfg["gen_ch"])

        self.r_freq = self.sreg(cfg["gen_ch"], "freq")   # the ASM register holding the freq field
        freq_reg = self.freq2reg(cfg["start"], gen_ch=cfg["gen_ch"], ro_ch=cfg["ro_ch"])
        self.set_pulse_registers(
            ch=cfg["gen_ch"], style="const",
            freq=freq_reg, phase=0, gain=cfg["gain"],
            length=self.us2cycles(cfg["pulse_length_us"], gen_ch=cfg["gen_ch"]),
        )
        self.freq_step_reg = self.freq2reg(cfg["step"], gen_ch=cfg["gen_ch"])
        self.synci(200)

    def body(self):
        self.trigger(ros=[self.cfg["ro_ch"]], pins=[], adc_trig_offset=self.cfg["adc_trig_offset"])
        self.pulse(ch=self.cfg["gen_ch"])
        self.wait_all()
        self.sync_all(self.us2cycles(1))

    def update(self):
        # runs once per outer-loop (expts) iteration, entirely in hardware —
        # this is what makes RAveragerProgram fast: no Python round-trip per point
        self.mathi(self.r_freq.page, self.r_freq.addr, self.r_freq.addr, "+", self.freq_step_reg)


# config = {
#     "gen_ch": 0, "ro_ch": 0,
#     "start": 90, "step": 0.2, "expts": 200,      # sweep 90 MHz to ~130 MHz
#     "gain": 20000, "pulse_length_us": 2, "readout_length": 400,
#     "adc_trig_offset": 100, "reps": 500, "rounds": 1,
# }
# prog = FreqSweepProgram(soccfg, config)
# expt_pts, avg_di, avg_dq = prog.acquire(soc, progress=True)
#
# import numpy as np, matplotlib.pyplot as plt
# mag = np.abs(avg_di[0] + 1j*avg_dq[0])
# plt.plot(expt_pts, mag); plt.xlabel("freq (MHz)"); plt.ylabel("|IQ|")


## 8.1 N-dimensional hardware sweeps (`NDAveragerProgram`)

### The core idea

`RAveragerProgram` sweeps exactly **one** variable in hardware. `NDAveragerProgram`
generalizes this to **several**, each nested inside the last -- still entirely in
hardware, zero Python round-trips per point.

### Think of it as nested `for` loops -- except the tProc runs them, not Python

```python
freq_sweep = QickSweep(self, freq_reg, start=90,   stop=130,   expts=41)
gain_sweep = QickSweep(self, gain_reg, start=2000, stop=30000, expts=15)

self.add_sweep(gain_sweep)   # added FIRST  -> INNERMOST loop, varies fastest
self.add_sweep(freq_sweep)   # added SECOND -> OUTERMOST sweep loop, varies slowest
```

is the hardware equivalent of:

```python
for rep in range(reps):              # always the outermost loop of all
    for freq in freq_points:         # last-added sweep
        for gain in gain_points:     # first-added sweep
            run_one_shot()
```

`freq_reg`/`gain_reg` are handles to the **actual pulse-parameter registers** --
`self.get_gen_reg(ch, "freq")` / `self.get_gen_reg(ch, "gain")` return the same registers
`set_pulse_registers`/`pulse()` already read from (see §6), so sweeping them changes what
the very next `pulse()` call actually plays. Note the unit difference: `"freq"` is a
recognized type, so `start`/`stop` there are in MHz (auto-converted); `"gain"` is *not* a
recognized type, so `start`/`stop` there must be raw DAC codes, same as the `gain=` you'd
pass to `set_pulse_registers` directly.

### Why this is fast

Between innermost iterations, no Python is involved at all -- the tProc just does the
hardware equivalent of `gain_register += gain_step` (one `mathi` instruction, from §5) and
loops back, entirely on-chip. Python only compiles this whole nested structure once, loads
it, starts it, and waits for the accumulated-buffer stream (§7.1) to complete.

### Getting the sweep points and result shape back

```python
freq_pts, gain_pts = prog.get_expt_pts()   # one array per add_sweep() call, in that order
avg_di, avg_dq = prog.acquire(soc)
```

The returned array's shape follows the nesting: after averaging over `reps`, the remaining
axes are ordered outermost -> innermost, i.e. `(freq_expts, gain_expts)` in this example --
so `avg_di[0]` is directly usable as a 2D map (`plt.imshow`), exactly as the loop nesting
above suggests.

### The building blocks, precisely

- **`QickSweep(prog, reg, start, stop, expts, label=None)`** — describes one swept axis:
  linearly step `reg` from `start` to `stop` over `expts` points.
- **`self.add_sweep(sweep)`** — registers a `QickSweep` with the program, in nesting order
  (first-added = innermost, as above). See `NDAveragerProgram.make_program()` if you want to
  see exactly how the loop nesting is built from this list.
- **`merge_sweeps([...])`** — combine several `QickSweep`s into one `AbsQickSweep` whose
  `update()`/`reset()` drive all of them together, useful when two registers must always
  step in lockstep (e.g. sweeping a generator's freq and its paired readout's freq as one
  axis, so they stay frequency-matched at every point).

### Worked example: frequency × gain 2D map


In [ ]:
# ⚠️ requires hardware — 2D hardware sweep: frequency (outer) x gain (inner).

from qick.averager_program import NDAveragerProgram, QickSweep

class NDSweepProgram(NDAveragerProgram):
    def initialize(self):
        cfg = self.cfg
        ch, ro_ch = cfg["gen_ch"], cfg["ro_ch"]
        self.declare_gen(ch=ch, nqz=1)
        self.declare_readout(ch=ro_ch, length=cfg["readout_length"],
                              freq=cfg["freq_start"], gen_ch=ch)

        # set up the pulse once with starting values -- the sweeps below will overwrite
        # the freq/gain registers in hardware between shots, no Python round-trip needed
        self.set_pulse_registers(
            ch=ch, style="const",
            freq=self.freq2reg(cfg["freq_start"], gen_ch=ch, ro_ch=ro_ch),
            phase=0, gain=cfg["gain_start"],
            length=self.us2cycles(cfg["pulse_length_us"], gen_ch=ch),
        )

        # grab handles to the SAME freq/gain registers set_pulse_registers just wrote
        freq_reg = self.get_gen_reg(ch, "freq")
        gain_reg = self.get_gen_reg(ch, "gain")

        # add_sweep order sets loop nesting: first-added = innermost (fastest-varying)
        self.add_sweep(QickSweep(self, gain_reg, cfg["gain_start"], cfg["gain_stop"], cfg["gain_expts"]))
        self.add_sweep(QickSweep(self, freq_reg, cfg["freq_start"], cfg["freq_stop"], cfg["freq_expts"]))
        # -> for each freq point (outer), gain sweeps fully (inner) before freq steps again

        self.synci(200)

    def body(self):
        self.trigger(ros=[self.cfg["ro_ch"]], pins=[], adc_trig_offset=self.cfg["adc_trig_offset"])
        self.pulse(ch=self.cfg["gen_ch"])
        self.wait_all()
        self.sync_all(self.us2cycles(1))


# config = {
#     "gen_ch": 0, "ro_ch": 0,
#     "freq_start": 90, "freq_stop": 130, "freq_expts": 41,
#     "gain_start": 2000, "gain_stop": 30000, "gain_expts": 15,
#     "pulse_length_us": 2, "readout_length": 400,
#     "adc_trig_offset": 100, "reps": 200, "soft_avgs": 1,
# }
# prog = NDSweepProgram(soccfg, config)
# freq_pts, gain_pts = prog.get_expt_pts()   # one array per added sweep, in add_sweep order
# avg_di, avg_dq = prog.acquire(soc, progress=True)
#
# import numpy as np, matplotlib.pyplot as plt
# # avg_di[0] shape follows loop nesting: (..., freq_expts, gain_expts) after averaging reps
# mag = np.abs(avg_di[0] + 1j*avg_dq[0])
# plt.imshow(mag, aspect="auto", origin="lower",
#            extent=[gain_pts[0], gain_pts[-1], freq_pts[0], freq_pts[-1]])
# plt.xlabel("gain (DAC code)"); plt.ylabel("freq (MHz)"); plt.colorbar(label="|IQ|")


## 9. Board characterization: measuring DAC output power (dBm / V)

Goal: sweep `gain` (and/or frequency), measure the *real* output power at the SMA
connector, and build a calibration curve — since `gain` alone (§6.1) doesn't tell you dBm.

### Step 1 — generate a CW tone
Use `mode="periodic"` so the generator repeats the pulse continuously after `pulse()`,
instead of firing once — a true CW tone rather than back-to-back short pulses.

### Step 2 — measure with an external instrument
Connect the DAC SMA output through a **known, fixed attenuator** (protects your instrument,
e.g. 10–20 dB) to a spectrum analyzer or RF power meter. Read the power in dBm, add back the
attenuator value → that's the real DAC output power for that `(freq, gain)` setting.

No spectrum analyzer? An oscilloscope works: measure `Vpp`, then for a sine tone
`Vrms = Vpp / (2*sqrt(2))`, and `P(dBm) = 10*log10((Vrms**2 / 50) / 1e-3)` (50 Ω system).

### Step 3 — sweep and log
Repeat across `gain` (expect roughly `P(dBm) ≈ 20*log10(gain/gain_max) + P_max`, flattening
near full scale) and across `freq` (to see the frequency response / Nyquist-zone roll-off).

### Alternative: DAC→ADC loopback (no external instrument)
Route the DAC output into an ADC input through a short coax + fixed attenuator, and use
`acquire_decimated()` (§7) to read raw samples, converting ADC codes to volts using the
ADC's own calibrated full-scale voltage. This gives relative/self-consistent calibration —
useful for balancing channels — but is only as absolute as your knowledge of the ADC's true
full-scale voltage. For real "power out the front panel," prefer Step 2.


In [ ]:
# ⚠️ requires hardware — CW tone generator for external power measurement.

from qick.averager_program import AveragerProgram

class CWToneProgram(AveragerProgram):
    def initialize(self):
        cfg = self.cfg
        self.declare_gen(ch=cfg["gen_ch"], nqz=cfg.get("nqz", 1))
        freq_reg = self.freq2reg(cfg["freq_MHz"], gen_ch=cfg["gen_ch"])
        self.set_pulse_registers(
            ch=cfg["gen_ch"], style="const",
            freq=freq_reg, phase=0, gain=cfg["gain"],
            length=self.us2cycles(cfg["length_us"], gen_ch=cfg["gen_ch"]),
            mode="periodic",     # <- key difference from ToneProgram: repeats continuously
        )
        self.synci(200)

    def body(self):
        self.pulse(ch=self.cfg["gen_ch"])
        self.wait_all()
        self.sync_all(self.us2cycles(1))


# gains_to_sweep = [1000, 5000, 10000, 15000, 20000, 25000, 30000]
# results_dBm = []   # fill in by hand from your spectrum analyzer / power meter reading
#
# for g in gains_to_sweep:
#     prog = CWToneProgram(soccfg, {"gen_ch": 0, "freq_MHz": 1000, "gain": g, "length_us": 10, "reps": 1})
#     prog.run_rounds(soc)     # starts the (periodic) tone; now go read your instrument
#     input(f"gain={g}: read the instrument, press Enter to continue...")


In [ ]:
# ✅ runs anywhere — dBm/voltage conversion helpers for the sweep above.
import numpy as np

def vpp_to_dbm(vpp, r_ohm=50):
    """Convert a sine-wave peak-to-peak voltage into dBm across r_ohm."""
    vrms = vpp / (2 * np.sqrt(2))
    p_watts = vrms**2 / r_ohm
    return 10 * np.log10(p_watts / 1e-3)

def dbm_to_vpp(dbm, r_ohm=50):
    """Inverse of vpp_to_dbm."""
    p_watts = 1e-3 * 10**(dbm / 10)
    vrms = np.sqrt(p_watts * r_ohm)
    return vrms * 2 * np.sqrt(2)

# sanity check: 0 dBm into 50 ohm is a well-known reference point (~632 mVpp)
print(f"0 dBm -> {dbm_to_vpp(0)*1000:.1f} mVpp")
print(f"{vpp_to_dbm(0.632):.2f} dBm <- 632 mVpp")


## 10. Glossary

| Term | Meaning |
|---|---|
| **RFSoC** | Xilinx/AMD chip combining ARM cores (PS), FPGA fabric (PL), and hardened RF-DAC/ADC tiles on one die |
| **tProc** | the small, deterministic soft-CPU inside the FPGA fabric that sequences pulses/readouts in real time |
| **PS / PL** | Processing System (ARM, runs Linux/Python) / Programmable Logic (FPGA fabric, runs the bitstream) |
| **DDS** | Direct Digital Synthesizer — generates a tone at a programmable frequency from a phase accumulator; frequencies are quantized to discrete steps because of finite accumulator bit width |
| **NCO / mixer** | Numerically Controlled Oscillator — a digital mixer that shifts a tone's frequency, used to reach frequencies beyond a single DDS's native range |
| **Nyquist zone** | Which multiple of fs/2 your target frequency falls in; affects available output power and requires `nqz=` to be set correctly |
| **fabric clock** | The clock rate a given generator/readout channel's digital logic runs at (distinct from the tProc's own clock and from the DAC/ADC sample rate) |
| **Envelope** | The I/Q amplitude-shape lookup table played back for `style="arb"`/`"flat_top"` pulses (Gaussian, DRAG, etc.) |
| **Gain (DAC code)** | Signed integer register value, a fraction of full scale (`get_maxv`) — not a physical unit; needs calibration to translate to dBm/V |
| **reps / rounds (soft_avgs)** | `reps` = repetitions done *in hardware* inside one tProc program run; `rounds`/`soft_avgs` = how many times Python re-runs and averages the whole program |
| **Accumulated vs decimated** | Accumulated = hardware-summed I/Q per readout window (fast, `acquire()`); decimated = raw un-summed ADC samples (slow, `acquire_decimated()`, your "scope trace") |
| **Threshold / angle** | Real-time hardware single-shot state discrimination: rotate I/Q by `angle`, then binarize against `threshold` |
| **PFB** | Polyphase filterbank — used by some multiplexed-readout firmware to separate multiple simultaneous tones into per-channel readouts |


## 11. Cheat sheet: where things live

| I want to... | Method | Defined in |
|---|---|---|
| See what hardware I have | `print(soccfg)` | `QickConfig.description()`, `qick_asm.py` |
| Configure a DAC channel | `declare_gen(...)` | `AbsQickProgram`, `qick_asm.py` |
| Configure an ADC channel | `declare_readout(...)` | `AbsQickProgram`, `qick_asm.py` |
| Define a pulse envelope | `add_gauss`/`add_DRAG`/`add_cosine`/`add_triangle`/`add_envelope` | `AbsQickProgram`, `qick_asm.py` |
| Set pulse freq/gain/phase/shape (v1) | `set_pulse_registers(...)` | `QickProgram`, `asm_v1.py` |
| Schedule the pulse in time | `pulse(ch=...)` | dynamically generated (`__getattr__`), `asm_v1.py` |
| Arm an ADC capture window | `trigger(ros=[...], ...)` | `QickProgram`, `asm_v1.py` |
| Convert MHz/deg/us to register units | `freq2reg`/`deg2reg`/`us2cycles` | `QickConfig`, `qick_asm.py` |
| Max DAC gain code for a channel | `get_maxv(ch)` | `QickConfig`, `qick_asm.py` |
| Run + collect averaged I/Q | `acquire(soc, ...)` | `AcquireMixin`, `qick_asm.py` |
| Run + collect raw ADC trace | `acquire_decimated(soc, ...)` | `AcquireMixin`, `qick_asm.py` |
| Run with no data collection | `run_rounds(soc, ...)` | `AcquireMixin`, `qick_asm.py` |
| Time axis for plotting a decimated trace | `get_time_axis(ro_index)` | `AcquireMixin`, `qick_asm.py` |


## 12. Where to go next

- **In this repo**: `src/libraries/qick/pyro4/` has working notebooks for the
  nameserver/server/client Pyro4 setup (`00_nameserver.ipynb`, `01_server.ipynb`,
  `02_client.ipynb`) — read these before setting up remote control for the lab.
  `src/libraries/qick/firmware/projects/` has the actual HDL source for each bitstream,
  if you ever need to understand or modify the firmware itself (not needed for normal
  experiment work).
- **Upstream project**: this library is vendored from the QICK open-source project
  (`openquantumhardware/qick` on GitHub) — useful for release notes, issues, and the
  broader user community, since everything here is a local copy of that codebase.
- **Next tutorial notebook to write**: single-shot qubit-state discrimination
  (`threshold`/`angle` in `acquire()`) and `merge_sweeps` for lockstep multi-register
  sweeps — only briefly mentioned here (§8.1 covers `NDAveragerProgram`/`QickSweep`
  themselves in full).
